# 30 squidpy lr support

**Purpose:** Evaluate transcriptomic support for prioritized LIANA axes with Squidpy.

**Original analysis notebook:** `27_squidpy_LR_validation_top_LIANA_axes_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.


# 46 — Squidpy validation of prioritized LIANA ligand–receptor axes (HVG8000 r0.6)

This notebook performs a lightweight independent validation of the final LIANA candidate communication axes.

It does **not** replace LIANA. LIANA remains the primary CCC analysis. This notebook adds:

1. expression-support cross-check for LIANA candidate ligand–receptor pairs;
2. optional Squidpy `ligrec` validation on a downsampled high-confidence object;
3. thesis-ready summaries for candidate LR axes.

Interpretation rule: results are **candidate communication support**, not functional proof.


In [ ]:

# ============================
# User settings
# ============================
from pathlib import Path
import os, sys, json, math, warnings, time, subprocess
warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURE_DIR = PROJECT_DIR / "figures" / "SQUIDPY_LR_validation_HVG8000_r0_6"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "SQUIDPY_LR_validation_HVG8000_r0_6"

FINAL_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"

LIANA_CANDIDATE_DETAILS = METADATA_DIR / "LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_source_target_details.tsv"
LIANA_CONDITION_PRESENCE = METADATA_DIR / "LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_condition_presence.tsv"
LIANA_CELLTYPE_COUNTS = METADATA_DIR / "LIANA_global_main_HVG8000_r0_6_clean_labels_celltype_counts_used.tsv"

# Squidpy validation is optional. If squidpy is unavailable, the notebook still produces expression-support validation.
RUN_SQUIDPY = True
SQUIDPY_N_PERMS = 100
SQUIDPY_THRESHOLD = 0.10
MAX_CELLS_PER_CELLTYPE = 2500
RANDOM_SEED = 42

# Candidate axes to validate. Keep these thesis-facing axes stable.
AXES_OF_INTEREST = [
    "PTN_PTPRZ1_NCL_SDC3_axis",
    "MDK_PTPRZ1_NCL_axis",
    "TIMP1_CD63_axis",
    "LGALS1_ITGB1_axis",
    "MIF_EGFR_CD74_axis",
    "VIM_CD44_axis",
    "VEGFA_angiogenesis_axis",
    "APP_CD74_axis",
]

MIN_EXPR_PROP_SUPPORT = 0.10


In [ ]:

# ============================
# Imports and helpers
# ============================
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    raise RuntimeError("scanpy/anndata are required in the active Python environment.") from e


def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def choose_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("None of the candidate paths exists:\n" + "\n".join(map(str, paths)))


def choose_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns were found: {candidates}. Available columns: {list(df.columns)[:50]}")
    return None


def clean_celltype_label(x):
    x = str(x)
    x = x.replace(" ", "_").replace("/", "_").replace("-", "_")
    x = x.replace("__", "_")
    return x


def is_complex_or_missing_gene(gene, var_names):
    # Squidpy custom interactions are most robust for single genes present in var_names.
    g = str(gene)
    if g not in var_names:
        return True
    return False


def to_dense_1d(x):
    if sp.issparse(x):
        return np.asarray(x.toarray()).ravel()
    return np.asarray(x).ravel()


def matrix_subset_mean_pct(X, gene_indices, cell_indices):
    # Returns pct>0 and mean for genes across a subset of cells.
    sub = X[cell_indices, :][:, gene_indices]
    if sp.issparse(sub):
        pct = np.asarray((sub > 0).mean(axis=0)).ravel()
        mean = np.asarray(sub.mean(axis=0)).ravel()
    else:
        pct = (sub > 0).mean(axis=0)
        mean = sub.mean(axis=0)
    return pct, mean


def safe_neglog10(x):
    x = pd.to_numeric(x, errors="coerce")
    return -np.log10(np.maximum(x, 1e-300))


In [ ]:

# ============================
# Load final object and LIANA candidates
# ============================
h5ad_path = FINAL_H5AD
if not h5ad_path.exists():
    raise FileNotFoundError(f"Required final malignant-state object not found: {h5ad_path}")
print("[READ]", h5ad_path)
adata = sc.read_h5ad(h5ad_path)
print(adata)

if not LIANA_CANDIDATE_DETAILS.exists():
    raise FileNotFoundError(f"Missing LIANA candidate details: {LIANA_CANDIDATE_DETAILS}")

cand = pd.read_csv(LIANA_CANDIDATE_DETAILS, sep="\t")
print("candidate details", cand.shape)
display(cand.head())

presence = pd.read_csv(LIANA_CONDITION_PRESENCE, sep="\t") if LIANA_CONDITION_PRESENCE.exists() else pd.DataFrame()
celltype_counts = pd.read_csv(LIANA_CELLTYPE_COUNTS, sep="\t") if LIANA_CELLTYPE_COUNTS.exists() else pd.DataFrame()

readiness = pd.DataFrame([
    {"item": "h5ad_path", "value": str(h5ad_path)},
    {"item": "n_cells", "value": adata.n_obs},
    {"item": "n_genes", "value": adata.n_vars},
    {"item": "candidate_detail_rows", "value": cand.shape[0]},
    {"item": "condition_presence_rows", "value": presence.shape[0]},
    {"item": "celltype_counts_rows", "value": celltype_counts.shape[0]},
])
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")


In [ ]:

# ============================
# Resolve obs columns and construct Squidpy/validation cell-type labels
# ============================
obs = adata.obs.copy()
condition_col = choose_col(obs, ["condition", "disease", "diagnosis"], required=False)
sample_col = choose_col(obs, ["sample_id", "sample", "Sample"], required=False)
state_col = choose_col(obs, ["condition_malignant_state", "malignant_state", "final_malignant_state"], required=False)
annotation_col = choose_col(obs, ["final_annotation_refined", "annotation_marker_based_HVG8000_r0_6", "annotation"], required=False)
compartment_col = choose_col(obs, ["final_compartment_major", "final_malignancy_group_copykat_marker"], required=False)

print("condition_col", condition_col)
print("sample_col", sample_col)
print("state_col", state_col)
print("annotation_col", annotation_col)
print("compartment_col", compartment_col)

# Prefer an already prepared LIANA label if it exists.
prepared_label_col = choose_col(obs, ["liana_celltype_main", "liana_celltype", "liana_main_celltype", "LIANA_celltype_main"], required=False)

if prepared_label_col is not None:
    adata.obs["squidpy_liana_celltype"] = adata.obs[prepared_label_col].astype(str).map(clean_celltype_label)
else:
    labels = pd.Series("exclude", index=adata.obs_names, dtype=object)
    if state_col is not None:
        st = adata.obs[state_col].astype(str).str.lower()
        labels[st.str.contains("opc") & st.str.contains("npc")] = "Malignant_OPC_NPC_like_malignant"
        labels[st.str.contains("hypoxia") | st.str.contains("stress")] = "Malignant_hypoxia_stress_malignant"
        labels[st.str.contains("ac") & st.str.contains("reactive")] = "Malignant_AC_like_reactive_malignant"
        labels[st.str.contains("neuronal") | st.str.contains("neural")] = "Malignant_NPC_neuronal_like_malignant"
        labels[st.str.contains("cycling")] = "Malignant_cycling_candidate"
    if annotation_col is not None:
        an = adata.obs[annotation_col].astype(str).str.lower()
        labels[an.str.contains("oligodendrocyte") | an.str.contains("myelin")] = "NonMalignant_oligodendrocyte_myelinating_reference_like"
        labels[an.str.contains("endothelial")] = "NonMalignant_endothelial_reference_like"
        labels[an.str.contains("mural") | an.str.contains("pericyte") | an.str.contains("fibroblast")] = "NonMalignant_mural_pericyte_fibroblast_like"
        labels[an.str.contains("b_plasma") | an.str.contains("plasma") | an.str.contains("b cell") | an.str.contains("b_cell")] = "NonMalignant_B_plasma_like"
    adata.obs["squidpy_liana_celltype"] = labels.map(clean_celltype_label)

# Restrict to cell types used in final LIANA global main if available.
if not celltype_counts.empty and "liana_celltype" in celltype_counts.columns:
    allowed_celltypes = set(celltype_counts["liana_celltype"].astype(str).map(clean_celltype_label))
elif "source_clean" in cand.columns and "target_clean" in cand.columns:
    allowed_celltypes = set(cand["source_clean"].astype(str).map(clean_celltype_label)).union(set(cand["target_clean"].astype(str).map(clean_celltype_label)))
else:
    allowed_celltypes = set(adata.obs["squidpy_liana_celltype"].unique()) - {"exclude"}

adata.obs["squidpy_liana_celltype"] = adata.obs["squidpy_liana_celltype"].astype(str)
mask_main = adata.obs["squidpy_liana_celltype"].isin(allowed_celltypes)

counts = adata.obs.loc[mask_main, "squidpy_liana_celltype"].value_counts().rename_axis("squidpy_liana_celltype").reset_index(name="n_cells")
counts["included_for_validation"] = True
write_tsv_replace(counts, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv")
display(counts)


In [ ]:

# ============================
# Build candidate LR interaction table for validation
# ============================
required_cols = ["candidate_axis", "ligand", "receptor", "source_clean", "target_clean"]
missing = [c for c in required_cols if c not in cand.columns]
if missing:
    raise KeyError(f"Missing required columns in LIANA candidate details: {missing}")

cand2 = cand.copy()
cand2["source_clean"] = cand2["source_clean"].astype(str).map(clean_celltype_label)
cand2["target_clean"] = cand2["target_clean"].astype(str).map(clean_celltype_label)
cand2["ligand"] = cand2["ligand"].astype(str)
cand2["receptor"] = cand2["receptor"].astype(str)
cand2["lr_pair"] = cand2["ligand"] + "→" + cand2["receptor"]

# Keep thesis-facing axes and source/target pairs present in validation labels.
cand2 = cand2[cand2["candidate_axis"].isin(AXES_OF_INTEREST)].copy()
cand2 = cand2[cand2["source_clean"].isin(allowed_celltypes) & cand2["target_clean"].isin(allowed_celltypes)].copy()

var_names = set(map(str, adata.var_names))
cand2["ligand_in_var"] = cand2["ligand"].isin(var_names)
cand2["receptor_in_var"] = cand2["receptor"].isin(var_names)
cand2["single_gene_pair_available"] = cand2["ligand_in_var"] & cand2["receptor_in_var"]

# Remove exact duplicates for validation.
validation_candidates = cand2.drop_duplicates(["candidate_axis", "source_clean", "target_clean", "ligand", "receptor"]).reset_index(drop=True)
write_tsv_replace(validation_candidates, METADATA_DIR / f"{OUT_PREFIX}_candidate_interactions.tsv")

inventory = (
    validation_candidates.groupby("candidate_axis")
    .agg(
        n_rows=("lr_pair", "size"),
        n_unique_lr=("lr_pair", "nunique"),
        n_source_target=("source_clean", lambda x: len(set(zip(validation_candidates.loc[x.index, "source_clean"], validation_candidates.loc[x.index, "target_clean"])))) ,
        n_single_gene_available=("single_gene_pair_available", "sum"),
    )
    .reset_index()
)
write_tsv_replace(inventory, METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_inventory.tsv")
display(inventory)


In [ ]:

# ============================
# Independent expression-support cross-check
# ============================
# This checks whether LIANA-prioritized ligand and receptor are expressed in the expected source/target cell types.
# It is not a ligand-receptor inference method, but it is a useful QC/validation layer.

X_counts = adata.layers["counts"] if "counts" in adata.layers else adata.X
all_genes = sorted(set(validation_candidates.loc[validation_candidates["single_gene_pair_available"], "ligand"]).union(
    set(validation_candidates.loc[validation_candidates["single_gene_pair_available"], "receptor"])
))
gene_to_idx = {g: int(np.where(adata.var_names.astype(str) == g)[0][0]) for g in all_genes if g in var_names}

# Precompute gene expression support by cell type and optionally condition.
rows = []
obs_ct = adata.obs["squidpy_liana_celltype"].astype(str)
conditions = [None]
if condition_col is not None:
    conditions += sorted(adata.obs[condition_col].astype(str).dropna().unique().tolist())

for cond in conditions:
    if cond is None:
        cond_mask = np.ones(adata.n_obs, dtype=bool)
        cond_name = "GLOBAL"
    else:
        cond_mask = (adata.obs[condition_col].astype(str).values == cond)
        cond_name = cond
    for ct in sorted(allowed_celltypes):
        idx = np.where(cond_mask & (obs_ct.values == ct))[0]
        if len(idx) == 0:
            continue
        gene_indices = [gene_to_idx[g] for g in all_genes if g in gene_to_idx]
        genes_here = [g for g in all_genes if g in gene_to_idx]
        if len(gene_indices) == 0:
            continue
        pct, mean = matrix_subset_mean_pct(X_counts, gene_indices, idx)
        for g, p, m in zip(genes_here, pct, mean):
            rows.append({
                "condition": cond_name,
                "celltype": ct,
                "gene": g,
                "n_cells": len(idx),
                "pct_expr": float(p),
                "mean_count": float(m),
            })

expr_support = pd.DataFrame(rows)
write_tsv_replace(expr_support, METADATA_DIR / f"{OUT_PREFIX}_gene_expression_support_by_celltype_condition.tsv")

# Map expression support back to candidate source-target interactions.
def get_support(cond_name, ct, gene):
    sub = expr_support[(expr_support["condition"] == cond_name) & (expr_support["celltype"] == ct) & (expr_support["gene"] == gene)]
    if sub.empty:
        return np.nan, np.nan, 0
    r = sub.iloc[0]
    return float(r["pct_expr"]), float(r["mean_count"]), int(r["n_cells"])

cand_rows = []
for _, r in validation_candidates.iterrows():
    if not r["single_gene_pair_available"]:
        continue
    for cond_name in conditions:
        cn = "GLOBAL" if cond_name is None else cond_name
        lpct, lmean, ns = get_support(cn, r["source_clean"], r["ligand"])
        rpct, rmean, nt = get_support(cn, r["target_clean"], r["receptor"])
        cand_rows.append({
            "condition": cn,
            "candidate_axis": r["candidate_axis"],
            "source_clean": r["source_clean"],
            "target_clean": r["target_clean"],
            "ligand": r["ligand"],
            "receptor": r["receptor"],
            "lr_pair": r["lr_pair"],
            "source_n_cells": ns,
            "target_n_cells": nt,
            "ligand_source_pct_expr": lpct,
            "receptor_target_pct_expr": rpct,
            "ligand_source_mean_count": lmean,
            "receptor_target_mean_count": rmean,
            "expression_support_pass_10pct": bool((lpct >= MIN_EXPR_PROP_SUPPORT) and (rpct >= MIN_EXPR_PROP_SUPPORT)) if np.isfinite(lpct) and np.isfinite(rpct) else False,
            "min_ligand_receptor_pct": float(np.nanmin([lpct, rpct])) if np.isfinite(lpct) or np.isfinite(rpct) else np.nan,
        })

candidate_expr = pd.DataFrame(cand_rows)
write_tsv_replace(candidate_expr, METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_by_condition.tsv")
global_expr = candidate_expr[candidate_expr["condition"] == "GLOBAL"].copy()
write_tsv_replace(global_expr, METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_global.tsv")

expr_summary = (
    candidate_expr.groupby(["condition", "candidate_axis"])
    .agg(
        n_candidate_interactions=("lr_pair", "size"),
        n_pass_10pct=("expression_support_pass_10pct", "sum"),
        frac_pass_10pct=("expression_support_pass_10pct", "mean"),
        median_min_pct=("min_ligand_receptor_pct", "median"),
        median_ligand_source_pct=("ligand_source_pct_expr", "median"),
        median_receptor_target_pct=("receptor_target_pct_expr", "median"),
    )
    .reset_index()
)
write_tsv_replace(expr_summary, METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_summary.tsv")
display(expr_summary.head(20))


In [ ]:

# ============================
# Optional Squidpy ligrec validation
# ============================
# This independently applies Squidpy's CellPhoneDB-like permutation framework to the LIANA candidate LR pairs.
# If squidpy is missing, the notebook will continue and record that status.

sq_status_rows = []
sq_means_long = pd.DataFrame()
sq_pvals_long = pd.DataFrame()

try:
    import squidpy as sq
    import inspect
    SQUIDPY_AVAILABLE = True
    sq_version = getattr(sq, "__version__", "unknown")
except Exception as e:
    SQUIDPY_AVAILABLE = False
    sq_version = None
    sq_import_error = repr(e)

if not RUN_SQUIDPY:
    sq_status_rows.append({"item": "squidpy_status", "value": "skipped_by_user_setting"})
elif not SQUIDPY_AVAILABLE:
    sq_status_rows.append({"item": "squidpy_status", "value": "not_installed_or_import_failed"})
    sq_status_rows.append({"item": "squidpy_error", "value": sq_import_error})
else:
    sq_status_rows.append({"item": "squidpy_status", "value": "available"})
    sq_status_rows.append({"item": "squidpy_version", "value": sq_version})
    try:
        rng = np.random.default_rng(RANDOM_SEED)
        use_mask = mask_main.copy()
        # Downsample per cell type for a lightweight validation.
        selected_indices = []
        obs_ct_vals = adata.obs["squidpy_liana_celltype"].astype(str).values
        for ct in sorted(allowed_celltypes):
            idx = np.where(use_mask & (obs_ct_vals == ct))[0]
            if len(idx) > MAX_CELLS_PER_CELLTYPE:
                idx = rng.choice(idx, size=MAX_CELLS_PER_CELLTYPE, replace=False)
            selected_indices.extend(idx.tolist())
        selected_indices = np.array(sorted(selected_indices))

        # Restrict to single-gene candidate ligand/receptor genes.
        sq_cands = validation_candidates[validation_candidates["single_gene_pair_available"]].copy()
        interactions = sorted(set(zip(sq_cands["ligand"].astype(str), sq_cands["receptor"].astype(str))))
        genes_for_sq = sorted(set([g for pair in interactions for g in pair if g in var_names]))
        if len(interactions) == 0 or len(genes_for_sq) == 0:
            raise RuntimeError("No valid single-gene candidate LR pairs available for Squidpy.")

        ad_sq = adata[selected_indices, genes_for_sq].copy()
        ad_sq.obs["squidpy_liana_celltype"] = ad_sq.obs["squidpy_liana_celltype"].astype("category")
        # Use raw counts if available, then normalize/log1p for Squidpy.
        if "counts" in adata.layers:
            # Need to subset the original counts to selected cells and selected genes.
            gene_idx = [int(np.where(adata.var_names.astype(str) == g)[0][0]) for g in genes_for_sq]
            ad_sq.X = adata.layers["counts"][selected_indices, :][:, gene_idx].copy()
        sc.pp.normalize_total(ad_sq, target_sum=1e4)
        sc.pp.log1p(ad_sq)

        # Build robust kwargs across Squidpy versions.
        sig = inspect.signature(sq.gr.ligrec)
        kwargs = dict(
            adata=ad_sq,
            cluster_key="squidpy_liana_celltype",
            interactions=interactions,
            threshold=SQUIDPY_THRESHOLD,
            copy=True,
            use_raw=False,
            n_perms=SQUIDPY_N_PERMS,
            seed=RANDOM_SEED,
        )
        if "corr_method" in sig.parameters:
            kwargs["corr_method"] = "fdr_bh"
        elif "fdr_method" in sig.parameters:
            kwargs["fdr_method"] = "fdr_bh"
        if "corr_axis" in sig.parameters:
            kwargs["corr_axis"] = "interactions"
        elif "fdr_axis" in sig.parameters:
            kwargs["fdr_axis"] = "interactions"
        if "numba_parallel" in sig.parameters:
            kwargs["numba_parallel"] = False

        print("[RUN Squidpy ligrec] cells", ad_sq.n_obs, "genes", ad_sq.n_vars, "interactions", len(interactions))
        res = sq.gr.ligrec(**kwargs)
        sq_status_rows.extend([
            {"item": "squidpy_run_status", "value": "completed"},
            {"item": "squidpy_n_cells", "value": ad_sq.n_obs},
            {"item": "squidpy_n_genes", "value": ad_sq.n_vars},
            {"item": "squidpy_n_candidate_interactions", "value": len(interactions)},
            {"item": "squidpy_n_perms", "value": SQUIDPY_N_PERMS},
            {"item": "squidpy_threshold", "value": SQUIDPY_THRESHOLD},
        ])

        def squidpy_table_to_long(tbl, value_col):
            out = []
            if tbl is None or len(tbl) == 0:
                return pd.DataFrame()
            for idx, row in tbl.iterrows():
                if isinstance(idx, tuple):
                    lig = str(idx[0])
                    rec = str(idx[1]) if len(idx) > 1 else ""
                else:
                    # Best-effort parsing of index.
                    s = str(idx)
                    if "_" in s:
                        parts = s.split("_")
                        lig, rec = parts[0], "_".join(parts[1:])
                    else:
                        lig, rec = s, ""
                for col, val in row.items():
                    if isinstance(col, tuple):
                        src = str(col[0])
                        tgt = str(col[1]) if len(col) > 1 else ""
                    else:
                        c = str(col)
                        if "|" in c:
                            src, tgt = c.split("|", 1)
                        elif "_" in c:
                            # This may be imperfect, but labels are also saved raw.
                            src, tgt = c, ""
                        else:
                            src, tgt = c, ""
                    out.append({
                        "ligand": lig,
                        "receptor": rec,
                        "lr_pair": f"{lig}→{rec}",
                        "source_clean": clean_celltype_label(src),
                        "target_clean": clean_celltype_label(tgt),
                        value_col: val,
                    })
            return pd.DataFrame(out)

        sq_means_long = squidpy_table_to_long(res.get("means"), "squidpy_mean") if isinstance(res, dict) else pd.DataFrame()
        sq_pvals_long = squidpy_table_to_long(res.get("pvalues"), "squidpy_pvalue") if isinstance(res, dict) else pd.DataFrame()

    except Exception as e:
        sq_status_rows.append({"item": "squidpy_run_status", "value": "error"})
        sq_status_rows.append({"item": "squidpy_error", "value": repr(e)})

sq_status = pd.DataFrame(sq_status_rows)
write_tsv_replace(sq_status, METADATA_DIR / f"{OUT_PREFIX}_squidpy_run_status.tsv")
write_tsv_replace(sq_means_long, METADATA_DIR / f"{OUT_PREFIX}_squidpy_means_long.tsv")
write_tsv_replace(sq_pvals_long, METADATA_DIR / f"{OUT_PREFIX}_squidpy_pvalues_long.tsv")
display(sq_status)


In [ ]:

# ============================
# Squidpy-candidate overlap summary
# ============================
if not sq_pvals_long.empty:
    merge_cols = ["source_clean", "target_clean", "ligand", "receptor", "lr_pair"]
    sqp = sq_pvals_long.copy()
    sqm = sq_means_long.copy() if not sq_means_long.empty else pd.DataFrame(columns=merge_cols + ["squidpy_mean"])
    sq_merge = sqp.merge(sqm, on=merge_cols, how="left")
    ov = validation_candidates.merge(sq_merge, on=merge_cols, how="left")
    ov["squidpy_tested"] = ov["squidpy_pvalue"].notna()
    ov["squidpy_pvalue_le_0_05"] = pd.to_numeric(ov["squidpy_pvalue"], errors="coerce") <= 0.05
    ov["squidpy_neglog10_p"] = safe_neglog10(ov["squidpy_pvalue"])
else:
    ov = validation_candidates.copy()
    ov["squidpy_tested"] = False
    ov["squidpy_pvalue_le_0_05"] = False
    ov["squidpy_pvalue"] = np.nan
    ov["squidpy_mean"] = np.nan
    ov["squidpy_neglog10_p"] = np.nan

write_tsv_replace(ov, METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_details.tsv")

overlap_summary = (
    ov.groupby("candidate_axis")
    .agg(
        n_candidate_interactions=("lr_pair", "size"),
        n_squidpy_tested=("squidpy_tested", "sum"),
        n_squidpy_p_le_0_05=("squidpy_pvalue_le_0_05", "sum"),
        frac_squidpy_p_le_0_05=("squidpy_pvalue_le_0_05", "mean"),
        min_squidpy_pvalue=("squidpy_pvalue", "min"),
        median_squidpy_mean=("squidpy_mean", "median"),
    )
    .reset_index()
)
write_tsv_replace(overlap_summary, METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_summary.tsv")
display(overlap_summary)


In [ ]:

# ============================
# Thesis-facing validation summary
# ============================
expr_global_summary = expr_summary[expr_summary["condition"] == "GLOBAL"].copy()
summary = expr_global_summary.merge(overlap_summary, on="candidate_axis", how="left")

# Simple interpretation label.
def interpret_row(r):
    expr_ok = r.get("frac_pass_10pct", 0) >= 0.5
    sq_tested = r.get("n_squidpy_tested", 0) > 0
    sq_ok = r.get("n_squidpy_p_le_0_05", 0) > 0
    if expr_ok and sq_tested and sq_ok:
        return "Supported by expression QC and at least one Squidpy ligrec signal"
    if expr_ok and sq_tested and not sq_ok:
        return "Expression-supported; Squidpy did not find significant support under this lightweight setting"
    if expr_ok and not sq_tested:
        return "Expression-supported; Squidpy not available/not completed"
    if not expr_ok and sq_ok:
        return "Partial expression support but Squidpy detected at least one signal; inspect manually"
    return "Weak/partial validation support under current settings; interpret cautiously"

summary["validation_interpretation"] = summary.apply(interpret_row, axis=1)
summary["suggested_reporting"] = summary["candidate_axis"].map({
    "PTN_PTPRZ1_NCL_SDC3_axis": "Prioritized LIANA axis; validate as recurrent candidate malignant/progenitor-like signaling.",
    "MDK_PTPRZ1_NCL_axis": "Prioritized LIANA axis; interpret as growth-factor-like candidate communication.",
    "TIMP1_CD63_axis": "Candidate ECM/remodeling/stress-associated axis.",
    "LGALS1_ITGB1_axis": "Candidate adhesion/ECM-associated axis.",
    "MIF_EGFR_CD74_axis": "Candidate inflammatory/growth signaling axis.",
    "VIM_CD44_axis": "Candidate mesenchymal/stress-associated axis.",
    "VEGFA_angiogenesis_axis": "Candidate angiogenic axis; condition-specific interpretation especially GBM/ndGBM/rGBM.",
    "APP_CD74_axis": "Candidate immune/inflammatory axis; interpret cautiously.",
}).fillna("Candidate LR axis; interpret as hypothesis-generating communication support.")

write_tsv_replace(summary, METADATA_DIR / f"{OUT_PREFIX}_thesis_validation_summary.tsv")
display(summary)


In [ ]:

# ============================
# Figures
# ============================
fig_manifest = []

# Barplot: fraction expression-supported by candidate axis.
try:
    plot_df = expr_global_summary.sort_values("frac_pass_10pct", ascending=True)
    plt.figure(figsize=(9, max(4, 0.45 * len(plot_df))))
    plt.barh(plot_df["candidate_axis"], plot_df["frac_pass_10pct"])
    plt.xlabel("Fraction of candidate source-target LR rows with ligand/receptor expression ≥ 10%")
    plt.title("Expression support for LIANA candidate axes")
    plt.tight_layout()
    fig_path = FIGURE_DIR / f"{OUT_PREFIX}_expression_support_candidate_axes_barplot.png"
    plt.savefig(fig_path, dpi=200)
    plt.show()
    fig_manifest.append({"figure": "expression_support_candidate_axes_barplot", "path": str(fig_path)})
except Exception as e:
    print("[FIG ERROR expression support]", repr(e))

# Barplot: Squidpy signals by candidate axis, if available.
try:
    if "n_squidpy_p_le_0_05" in overlap_summary.columns and overlap_summary["n_squidpy_tested"].sum() > 0:
        plot_df = overlap_summary.sort_values("n_squidpy_p_le_0_05", ascending=True)
        plt.figure(figsize=(9, max(4, 0.45 * len(plot_df))))
        plt.barh(plot_df["candidate_axis"], plot_df["n_squidpy_p_le_0_05"])
        plt.xlabel("Number of candidate rows with Squidpy p ≤ 0.05")
        plt.title("Lightweight Squidpy validation of LIANA candidate axes")
        plt.tight_layout()
        fig_path = FIGURE_DIR / f"{OUT_PREFIX}_squidpy_candidate_axes_barplot.png"
        plt.savefig(fig_path, dpi=200)
        plt.show()
        fig_manifest.append({"figure": "squidpy_candidate_axes_barplot", "path": str(fig_path)})
except Exception as e:
    print("[FIG ERROR squidpy]", repr(e))

fig_manifest_df = pd.DataFrame(fig_manifest)
write_tsv_replace(fig_manifest_df, METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv")


In [ ]:

# ============================
# Output manifest
# ============================
expected_outputs = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_inventory.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_gene_expression_support_by_celltype_condition.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_by_condition.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_global.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_expression_support_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_run_status.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_means_long.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_pvalues_long.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_details.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_squidpy_candidate_overlap_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_thesis_validation_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
]
manifest = []
for p in expected_outputs:
    manifest.append({
        "path": str(p),
        "exists": Path(p).exists(),
        "size_mb": round(Path(p).stat().st_size / 1024 / 1024, 3) if Path(p).exists() else 0,
    })
manifest = pd.DataFrame(manifest)
write_tsv_replace(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
display(manifest)
print("[DONE] Squidpy/lightweight LR validation notebook completed.")
